# Taller Pandas: 120 años de historia olímpica

**Curso:** Programación II, Ciencia de Datos
**Grupo:** taller-pandas-olimpiadas-grupoX

En este notebook analizamos los datos de Kaggle *120 years of Olympic history: athletes and results* (`athlete_events.csv` y `noc_regions.csv`). Seguimos el orden de los 14 puntos del taller: carga, descripción, nuevas variables, fusión, datos faltantes, preguntas con `groupby` y tablas de contingencia, gráficos y conclusiones.

> Cada fila de `athlete_events.csv` es **un atleta en una prueba de una edición de los Juegos**, no un atleta único. Lo tenemos en cuenta en todo el análisis.

Los archivos originales deben estar en la carpeta `data/` (ver `data/README.md`). Las tablas se guardan en `outputs/tablas/` y los gráficos en `outputs/graficos/`.

In [1]:
# Librerías y rutas del proyecto
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA = "../data"
TABLAS = "../outputs/tablas"
GRAFICOS = "../outputs/graficos"
os.makedirs(TABLAS, exist_ok=True)
os.makedirs(GRAFICOS, exist_ok=True)

## 1. Carga de los datos

Cargamos cada archivo en un dataframe distinto: `atletas` y `regiones`.

In [2]:
# Punto 1: carga de cada dataset en un dataframe distinto
atletas = pd.read_csv(f"{DATA}/athlete_events.csv")
regiones = pd.read_csv(f"{DATA}/noc_regions.csv")

print("atletas:", atletas.shape)
print("regiones:", regiones.shape)
atletas.head()

atletas: (271116, 15)
regiones: (230, 3)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [3]:
regiones.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


`atletas` tiene **271.116 filas y 15 columnas**, y `regiones` tiene **230 filas y 3 columnas**.

## 2. Descripción de las variables

**athlete_events.csv** (`atletas`)

| Variable | Tipo | Descripción |
|---|---|---|
| ID | entero | Identificador único de cada atleta |
| Name | texto | Nombre del atleta |
| Sex | categórica | Sexo: M (hombre) o F (mujer) |
| Age | numérica | Edad en años al momento de la prueba |
| Height | numérica | Estatura en centímetros |
| Weight | numérica | Peso en kilogramos |
| Team | texto | Nombre del equipo con el que compitió |
| NOC | categórica | Código de tres letras del Comité Olímpico Nacional |
| Games | categórica | Edición de los Juegos (año + temporada) |
| Year | entero | Año de la edición |
| Season | categórica | Temporada: Summer (verano) o Winter (invierno) |
| City | texto | Ciudad sede |
| Sport | categórica | Deporte |
| Event | categórica | Prueba específica dentro del deporte |
| Medal | categórica | Gold, Silver, Bronze o vacío (NA) si no ganó |

**noc_regions.csv** (`regiones`)

| Variable | Tipo | Descripción |
|---|---|---|
| NOC | categórica | Código del Comité Olímpico Nacional (llave para unir) |
| region | texto | País o región a la que corresponde el NOC |
| notes | texto | Notas aclaratorias (por ejemplo, nombres históricos como Bohemia o Yugoslavia) |

In [4]:
# Punto 2: tipos de dato y valores no nulos de cada variable
atletas.info()
print()
regiones.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB

<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   NOC     230 non-null    s

Las variables numéricas son `ID`, `Age`, `Height`, `Weight` y `Year` (aunque `ID` y `Year` funcionan más como identificadores que como medidas). Todo lo demás es texto o categoría. Desde ya se ve que `Age`, `Height`, `Weight` y `Medal` tienen menos valores no nulos que el total de filas, y en `regiones` faltan algunos valores de `region` (3) y casi todas las `notes`.

## 3. Estadísticas básicas y comparación entre dataframes

In [5]:
# Punto 3: estadísticas básicas de las variables numéricas de atletas
atletas.describe().round(2)

,ID,Age,Height,Weight,Year
count,271116.00,261642.00,210945.00,208241.00,271116.00
mean,68248.95,25.56,175.34,70.70,1978.38
std,39022.29,6.39,10.52,14.35,29.88
min,1.00,10.00,127.00,25.00,1896.00
25%,34643.00,21.00,168.00,60.00,1960.00
50%,68205.00,24.00,175.00,70.00,1988.00
75%,102097.25,28.00,183.00,79.00,2002.00
max,135571.00,97.00,226.00,214.00,2016.00


In [6]:
# Estadísticas de las variables categóricas de atletas
atletas.describe(exclude="number")

,Name,Sex,Team,NOC,Games,Season,City,Sport,Event,Medal
count,271116,271116,271116,271116,271116,271116,271116,271116,271116,39783
unique,134732,2,1184,230,51,2,42,66,765,3
top,Robert Tait McKenzie,M,United States,USA,2000 Summer,Summer,London,Athletics,Football Men's Football,Gold
freq,58,196594,17847,18853,13821,222552,22426,38624,5733,13372


In [7]:
# Estadísticas de regiones
regiones.describe(include="all")

,NOC,region,notes
count,230,227,21
unique,230,206,21
top,AFG,Germany,Netherlands Antilles
freq,1,4,1


In [8]:
# Comparación entre dataframes a través de la llave NOC
noc_atletas = set(atletas["NOC"].unique())
noc_regiones = set(regiones["NOC"].unique())

print("NOC distintos en atletas:", len(noc_atletas))
print("NOC distintos en regiones:", len(noc_regiones))
print("NOC en atletas que no están en regiones:", sorted(noc_atletas - noc_regiones))
print("NOC en regiones que no están en atletas:", sorted(noc_regiones - noc_atletas))
print("Regiones distintas:", regiones["region"].nunique())
print("Equipos (Team) distintos en atletas:", atletas["Team"].nunique())

NOC distintos en atletas: 230
NOC distintos en regiones: 230
NOC en atletas que no están en regiones: ['SGP']
NOC en regiones que no están en atletas: ['SIN']
Regiones distintas: 206
Equipos (Team) distintos en atletas: 1184


**Interpretación**

- La edad promedio por registro es de **25,6 años** (mediana 24), con un rango muy amplio: de 10 a 97 años.
- La estatura promedio es de **175,3 cm** y el peso promedio de **70,7 kg**. El 50 % central de los atletas mide entre 168 y 183 cm.
- Los años van de **1896 a 2016**, pero la mediana es 1988, lo que significa que la mitad de los registros son de los últimos ~30 años.
- Hay **134.732 nombres distintos**, 1.184 equipos, 230 NOC, 51 ediciones, 66 deportes y 765 pruebas. El 72,5 % de los registros son hombres (196.594 de 271.116).
- **Comparación:** ambos dataframes tienen 230 NOC, pero no son exactamente los mismos: `SGP` (Singapur) está en atletas y no en regiones, y `SIN` está en regiones y no en atletas. Además, los 230 NOC se agrupan en solo **206 regiones**, porque varios códigos históricos apuntan al mismo país (por ejemplo, GER, FRG, GDR y SAA corresponden a Germany). `atletas` tiene mucho más detalle (1.184 valores de `Team`) que `regiones`, que solo traduce códigos.